# ⚡ Arbitrage GPU Matcher v4
## Binary-only | bge-m3 + bge-reranker-v2-m3 | FAISS GPU

### Key improvements over v3:
1. **Binary-only filter** — drops all multi-outcome markets before encoding (only 2-outcome YES/NO markets can produce arbitrage)
2. **Sports noise filter** — removes exact-score / map / round markets that can never match prediction markets
3. **BAAI/bge-m3** — newer bi-encoder, tops MTEB leaderboard, better on political/financial text
4. **BAAI/bge-reranker-v2-m3** — purpose-built reranker replacing the repurposed NLI model
5. **FAISS GPU index** — approximate nearest-neighbour search, 10-50× faster than brute-force cosine at 50k+ scale
6. **Batch size 256** — fully utilises T4's 16 GB VRAM

### Instructions:
1. **Runtime > Change runtime type** → T4 GPU
2. **Runtime > Run all**

In [ ]:
# 1. Install packages
!pip install -q sentence-transformers torch httpx pydantic dateparser spacy faiss-gpu
!python -m spacy download en_core_web_sm -q

import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import httpx
import time, re, json, numpy as np
import dateparser
import spacy
from datetime import datetime
from typing import List, Dict, Any, Set, Tuple, Optional

print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

nlp = spacy.load('en_core_web_sm')
print('Setup complete!')

In [ ]:
# 2. Load models
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# bge-m3: newer generation, multilingual, tops MTEB — replaces bge-large-en-v1.5
print('Loading Bi-Encoder (BAAI/bge-m3)...')
bi_model = SentenceTransformer('BAAI/bge-m3', device=DEVICE)

# bge-reranker-v2-m3: purpose-built reranker — replaces repurposed NLI DeBERTa
print('Loading Reranker (BAAI/bge-reranker-v2-m3)...')
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=DEVICE)

print('Models loaded.')

In [ ]:
# 3. Fetch and filter markets
LOCAL_BACKEND_URL = "https://debian-1.tail8d8abc.ts.net/api"

# Patterns that indicate non-arbitrageable sports/gaming markets
NOISE_PATTERNS = re.compile(
    r'exact score|\bmap \d+\b|\bround \d+\b|\bset \d+\b|\bhalf \d+\b'
    r'|over/under|o/u \d|\+/-|handicap|total kills|total goals'
    r'|first blood|first dragon|odd/even|both teams to score',
    re.IGNORECASE
)

def fetch_markets():
    print(f'Fetching from {LOCAL_BACKEND_URL}/raw-markets...')
    resp = httpx.get(f'{LOCAL_BACKEND_URL}/raw-markets', timeout=60.0)
    resp.raise_for_status()
    data = resp.json()
    print(f'Raw markets received: {len(data):,}')

    # Deduplicate
    seen, unique = set(), []
    for m in data:
        mid = m.get('id') or m.get('marketUrl')
        if mid not in seen:
            seen.add(mid)
            unique.append(m)
    print(f'After dedup: {len(unique):,}')

    # BINARY ONLY — arbitrage requires exactly 2 outcomes (YES/NO)
    binary = [m for m in unique if m.get('isBinary', True) and m.get('outcomeCount', 2) <= 2]
    print(f'Binary markets only: {len(binary):,} (dropped {len(unique)-len(binary):,} multi-outcome)')

    # Drop sports noise that will never match prediction markets
    clean = [m for m in binary if not NOISE_PATTERNS.search(m.get('title', ''))]
    print(f'After noise filter: {len(clean):,} (dropped {len(binary)-len(clean):,} sports/gaming noise)')

    by_platform = {}
    for m in clean:
        by_platform.setdefault(m['platform'], []).append(m)
    for plat, mkts in by_platform.items():
        print(f'  {plat}: {len(mkts):,} binary markets')

    return clean

all_markets = fetch_markets()
print(f'\nTotal markets for matching: {len(all_markets):,}')

In [ ]:
# 4. Compatibility filters (date, number, entity)

def extract_dates(text: str) -> Set[Tuple[int, int, int]]:
    dates = set()
    patterns = [
        r'\b(\d{1,2})/(\d{1,2})/(\d{4})\b',
        r'\b(\d{4})-(\d{1,2})-(\d{1,2})\b',
        r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{1,2})(?:st|nd|rd|th)?(?:,?\s+(\d{4}))?\b',
    ]
    for pattern in patterns:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            try:
                parsed = dateparser.parse(match.group(0), settings={'PREFER_DATES_FROM': 'future'})
                if parsed:
                    dates.add((parsed.year, parsed.month, parsed.day))
            except:
                pass
    return dates

def extract_numbers(text: str) -> Set[float]:
    return {float(m) for m in re.findall(r'\d+(?:\.\d+)?', text) if not (2020 <= float(m) <= 2030)}

def are_compatible(title_a: str, title_b: str) -> Tuple[bool, str]:
    dates_a, dates_b = extract_dates(title_a), extract_dates(title_b)
    if dates_a and dates_b and dates_a.isdisjoint(dates_b):
        return False, 'Date mismatch'
    nums_a, nums_b = extract_numbers(title_a), extract_numbers(title_b)
    if nums_a and nums_b:
        if not any(any(abs(a - b) < 0.5 for b in nums_b) for a in nums_a):
            return False, 'Number mismatch'
    doc_a = nlp(title_a[:200].lower())
    doc_b = nlp(title_b[:200].lower())
    ents_a = {e.text for e in doc_a.ents if e.label_ in ('PERSON', 'ORG', 'GPE')}
    ents_b = {e.text for e in doc_b.ents if e.label_ in ('PERSON', 'ORG', 'GPE')}
    if ents_a and ents_b and ents_a.isdisjoint(ents_b):
        return False, 'Entity mismatch'
    return True, 'Compatible'

print('Compatibility filters ready.')

In [ ]:
# 5. FAISS GPU matching — bge-m3 bi-encoder + bge-reranker-v2-m3

def compute_pair_arb(ma, mb):
    p1_a = ma.get('bestBid', ma['yesPrice'])
    p1_b = 1 - mb.get('bestAsk', mb['yesPrice'])
    cost1 = p1_a + p1_b
    roi1 = ((1 - cost1) / max(0.01, cost1) * 100) if 0.05 < cost1 < 1 else -100
    p2_a = 1 - ma.get('bestAsk', ma['yesPrice'])
    p2_b = mb.get('bestBid', mb['yesPrice'])
    cost2 = p2_a + p2_b
    roi2 = ((1 - cost2) / max(0.01, cost2) * 100) if 0.05 < cost2 < 1 else -100
    if roi1 >= roi2:
        return {'roi': min(1000, roi1), 'cost': cost1, 'scenario': 1}
    return {'roi': min(1000, roi2), 'cost': cost2, 'scenario': 2}

def match_markets(markets, top_k=2000, min_roi=0.1, faiss_top_k=50):
    by_platform = {}
    for m in markets:
        by_platform.setdefault(m['platform'], []).append(m)
    platforms = list(by_platform.keys())
    print(f'Platforms: {platforms}')

    # Encode all platforms with bge-m3
    # bge-m3 uses empty prefix (unlike bge-large which needed 'Represent this sentence:')
    plat_embeddings = {}
    for plat in platforms:
        titles = [m['title'] for m in by_platform[plat]]
        print(f'  Encoding {len(titles):,} markets for {plat}...')
        embs = bi_model.encode(
            titles, convert_to_tensor=False, batch_size=256,
            normalize_embeddings=True, show_progress_bar=True
        ).astype('float32')
        plat_embeddings[plat] = embs
        print(f'  {plat} encoded: shape={embs.shape}')

    dim = list(plat_embeddings.values())[0].shape[1]

    candidates = []
    for i in range(len(platforms)):
        pa = platforms[i]
        embs_a = plat_embeddings[pa]
        mkts_a = by_platform[pa]

        for j in range(i + 1, len(platforms)):
            pb = platforms[j]
            embs_b = plat_embeddings[pb]
            mkts_b = by_platform[pb]

            # Build FAISS index on the smaller set, query with the larger
            if len(embs_a) < len(embs_b):
                index_embs, index_mkts = embs_a, mkts_a
                query_embs, query_mkts = embs_b, mkts_b
            else:
                index_embs, index_mkts = embs_b, mkts_b
                query_embs, query_mkts = embs_a, mkts_a

            print(f'\n  Building FAISS index for {pb if len(embs_a)>=len(embs_b) else pa} ({len(index_embs):,} vectors)...')
            res = faiss.StandardGpuResources()
            index_flat = faiss.IndexFlatIP(dim)  # inner product = cosine for normalised vecs
            index_gpu = faiss.index_cpu_to_gpu(res, 0, index_flat)
            index_gpu.add(index_embs)

            k = min(faiss_top_k, len(index_embs))
            print(f'  Searching top-{k} neighbours for {len(query_embs):,} queries...')
            scores, indices = index_gpu.search(query_embs, k)

            pair_count = 0
            for qi, (score_row, idx_row) in enumerate(zip(scores, indices)):
                qm = query_mkts[qi]
                for score, idx in zip(score_row, idx_row):
                    if score < 0.62 or idx < 0:
                        break
                    im = index_mkts[idx]
                    compatible, reason = are_compatible(qm['title'], im['title'])
                    if not compatible:
                        continue
                    ma, mb = (qm, im) if qm['platform'] == pa else (im, qm)
                    arb = compute_pair_arb(ma, mb)
                    if arb['roi'] >= min_roi:
                        candidates.append((ma, mb, arb['roi'], float(score), reason))
                        pair_count += 1

            print(f'  {pa} x {pb}: {pair_count:,} candidates (cosine >= 0.62)')

    if not candidates:
        print('No candidates found.')
        return []

    # Deduplicate and take top candidates for reranking
    seen_pairs, deduped = set(), []
    for c in sorted(candidates, key=lambda x: (x[3], x[2]), reverse=True):
        key = tuple(sorted([c[0].get('id',''), c[1].get('id','')]))
        if key not in seen_pairs:
            seen_pairs.add(key)
            deduped.append(c)

    rerank_pool = deduped[:top_k * 3]  # rerank 3× the target, keep best top_k
    print(f'\nReranking {len(rerank_pool):,} candidates with bge-reranker-v2-m3...')

    pairs_for_rerank = [[c[0]['title'], c[1]['title']] for c in rerank_pool]
    rerank_scores = reranker.predict(pairs_for_rerank, show_progress_bar=True)

    final_pairs = []
    for i, (score, cand) in enumerate(zip(rerank_scores, rerank_pool)):
        ma, mb, roi, bi_score, compat_reason = cand
        # bge-reranker outputs a single relevance score (higher = more relevant)
        if score > 0.5:
            final_pairs.append({
                'marketA': {
                    'id': ma.get('id'), 'platform': ma['platform'],
                    'title': ma['title'], 'marketUrl': ma.get('marketUrl', ''),
                    'yesPrice': ma['yesPrice'],
                    'noPrice': ma.get('noPrice', round(1 - ma['yesPrice'], 4)),
                    'endDate': ma.get('endDate'),
                },
                'marketB': {
                    'id': mb.get('id'), 'platform': mb['platform'],
                    'title': mb['title'], 'marketUrl': mb.get('marketUrl', ''),
                    'yesPrice': mb['yesPrice'],
                    'noPrice': mb.get('noPrice', round(1 - mb['yesPrice'], 4)),
                    'endDate': mb.get('endDate'),
                },
                'roi': roi,
                'matchScore': round(float(score) * 100, 1),
                'biEncoderScore': round(bi_score, 4),
                'matchReason': compat_reason,
                'isVerified': float(score) > 0.80,
            })

    final_pairs.sort(key=lambda x: (x['isVerified'], x['matchScore'], x['roi']), reverse=True)
    print(f'Done. {len(final_pairs):,} matches pass reranker threshold.')
    return final_pairs[:top_k]

found_pairs = match_markets(all_markets, top_k=2000, min_roi=0.1, faiss_top_k=50)
print(f'\nFinal: {len(found_pairs):,} verified arbitrage pairs.')

In [ ]:
# 6. Send results back
def post_results(pairs):
    if not pairs:
        print('No pairs to send.')
        return
    print(f'Sending {len(pairs):,} matches to backend...')
    batch_size = 500
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        clear = (i == 0)
        try:
            resp = httpx.post(
                f'{LOCAL_BACKEND_URL}/cloud-results?clear={str(clear).lower()}',
                json=batch, timeout=90.0
            )
            resp.raise_for_status()
            print(f'  Batch {i//batch_size + 1}: {len(batch)} sent')
        except Exception as e:
            print(f'  Batch {i//batch_size + 1} failed: {e}')
    print('Done.')

post_results(found_pairs)

In [ ]:
# 7. Preview top results
print('\n' + '='*90)
print(f'TOP {min(10, len(found_pairs))} ARBITRAGE OPPORTUNITIES')
print('='*90)
for i, p in enumerate(found_pairs[:10], 1):
    verified = '✅ VERIFIED' if p['isVerified'] else '⚠️  Review'
    print(f"\n#{i} | ROI: {p['roi']:.2f}% | Match: {p['matchScore']}% | {verified}")
    print(f"  [A] {p['marketA']['platform']:12} | {p['marketA']['title']}")
    print(f"      YES: {p['marketA']['yesPrice']:.3f}  NO: {p['marketA']['noPrice']:.3f}")
    print(f"  [B] {p['marketB']['platform']:12} | {p['marketB']['title']}")
    print(f"      YES: {p['marketB']['yesPrice']:.3f}  NO: {p['marketB']['noPrice']:.3f}")